# Coactivation graph threshold sensitivity — all models

Run the same threshold sweep for every architecture in **M&Ms-2** and **ACDC**. The combined pandas DataFrame is **`df`**, with one row per dataset, model, and threshold. Plots appear inline only: this notebook does not export CSVs, matrices, or figures.

Existing `<model>_scag.npz` archives are reused when available. Otherwise, the trained checkpoints and processed data are loaded using the existing SCAG pipeline helpers. Concept labels are calibrated on training images; coactivation graphs use the official validation + test cohort. The test split is therefore not untouched by this analysis.

In [ ]:
%matplotlib inline
import gc
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from scag.activations import extract_activations
from scag.config import SCAGConfig
from scag.data import build_graph_analysis_cohort, build_loaders, load_datasets, make_split_safe, phase_encode_segments
from scag.matrices import compute_R
from scag.models import build_model
from scag.semantics import compute_neuron_concept_matrix
from threshold_analysis import analyze_thresholds

torch.set_num_threads(8)

## Settings

The default sweep includes the current cutoff **0.35**. Edit `datasets` or `thresholds` here to rerun a smaller selection. All configured architectures are included automatically; no training, permutation tests, or ablation are run.

In [ ]:
datasets = ["mnms2", "acdc"]
thresholds = [0.1, 0.2, 0.3, 0.35, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
seed = 42

print("Thresholds:", thresholds)
for dataset in datasets:
    cfg = SCAGConfig(dataset=dataset)
    print(f"{dataset}: {len(cfg.architectures)} models, device={cfg.device}")

## Run every model

Only activation extraction, training concept calibration, coactivation computation, and the threshold sweep are needed. Dataset loaders are reused within each dataset, and each model is released before loading the next. Errors are collected in `failures_df` so one failed model does not prevent the others from running.

In [ ]:
frames = []
failures = []
df = pd.DataFrame()
started = time.perf_counter()

for dataset in datasets:
    cfg = SCAGConfig(dataset=dataset)
    loaders = None
    for tag in cfg.architectures:
        model_started = time.perf_counter()
        print(f"\n{'=' * 60}\n{dataset} / {tag}")
        try:
            archive = cfg.output_dir / f"{tag}_scag.npz"
            if archive.is_file():
                with np.load(archive, allow_pickle=False) as saved:
                    R_eval, X, labeled = saved["R_eval"], saved["X"], saved["labeled"]
                print("Using existing SCAG matrices:", archive)
            else:
                if loaders is None:
                    train_dataset, val_dataset, test_dataset, _ = load_datasets(cfg)
                    eval_dataset = make_split_safe(train_dataset, build_graph_analysis_cohort(val_dataset, test_dataset, cfg))
                    loaders = build_loaders(train_dataset, eval_dataset, batch_size=cfg.batch_size, num_workers=cfg.num_workers)
                train_loader, eval_loader = loaders
                model, hook_target, layer_name = build_model(tag, cfg)
                model = model.to(cfg.device).eval()
                train_acts, train_segments = extract_activations(model, train_loader, hook_target, cfg.device)
                X, labeled = compute_neuron_concept_matrix(train_acts, phase_encode_segments(train_segments), num_concepts=cfg.num_concepts, alpha=cfg.alpha, eta=cfg.eta)
                del train_acts, train_segments
                eval_acts, eval_segments = extract_activations(model, eval_loader, hook_target, cfg.device)
                R_eval = compute_R(eval_acts)
                del model, hook_target, eval_acts, eval_segments
            table = pd.DataFrame(analyze_thresholds(R_eval, X, labeled, thresholds=thresholds, seed=seed))
            table.insert(0, "model", tag)
            table.insert(0, "dataset", dataset)
            frames.append(table)
            df = pd.concat(frames, ignore_index=True)
            print(f"Completed {dataset}/{tag} in {(time.perf_counter() - model_started) / 60:.1f} min; {len(df)} rows so far.")
        except Exception as error:
            failures.append({"dataset": dataset, "model": tag, "error": f"{type(error).__name__}: {error}"})
            print(f"FAILED {dataset}/{tag}: {error}")
        finally:
            for name in ("model", "hook_target", "train_acts", "train_segments", "eval_acts", "eval_segments", "R_eval", "X", "labeled"):
                globals().pop(name, None)
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            if torch.backends.mps.is_available():
                torch.mps.empty_cache()

failures_df = pd.DataFrame(failures, columns=["dataset", "model", "error"])
expected_models = sum(len(SCAGConfig(dataset=dataset).architectures) for dataset in datasets)
completed_models = df[["dataset", "model"]].drop_duplicates().shape[0] if not df.empty else 0
print(f"\nCompleted {completed_models}/{expected_models} models in {(time.perf_counter() - started) / 60:.1f} min.")
if not failures_df.empty:
    display(failures_df)
assert not df.empty, "No models completed; see failures_df for the errors."
assert len(df) == completed_models * len(thresholds), "A completed model is missing threshold rows."
assert not df.duplicated(["dataset", "model", "threshold"]).any(), "Duplicate model/threshold rows."

## Combined results

`df` contains all results; for example, `df[df["threshold"] == 0.35]` selects the current threshold, and `df.query("dataset == 'acdc' and model == 'resnet18'")` selects one model.

Density and largest-component fraction use the **original neuron count**. Mean degree and the node/component/community counts use the graph **after isolates are removed**. AMI compares Louvain communities against dominant concepts on retained labeled neurons; check `n_labeled` when comparing thresholds. Undefined modularity or AMI is `NaN`. Louvain uses a single fixed seed, so these curves are descriptive rather than significance tests.

In [ ]:
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(df)

print("Threshold list:", df["threshold"].drop_duplicates().tolist())

## Threshold curves

Each line is one model. The dotted vertical line marks the pipeline cutoff, `tau = 0.35`. These figures are displayed inline and are not saved separately.

In [ ]:
metrics = {
    "n_edges": "Number of edges",
    "density": "Edge density (all neurons)",
    # "largest_component_fraction": "Largest component / all neurons",
    "n_comm": "Number of communities",
    # "modularity": "Weighted modularity",
    # "ami": "Semantic AMI",
}

for dataset in datasets:
    subset = df[df["dataset"] == dataset]
    if subset.empty:
        continue
    fig, axes = plt.subplots(1, 3, figsize=(14, 5))
    for ax, (metric, label) in zip(axes.flat, metrics.items()):
        for tag, group in subset.groupby("model", sort=False):
            group = group.sort_values("threshold")
            ax.plot(group["threshold"], group[metric], marker="o", markersize=3, linewidth=1.4, label=tag)
        ax.axvline(SCAGConfig(dataset=dataset).tau, color="black", linestyle=":", linewidth=1)
        ax.set(xlabel="Coactivation threshold (tau)", ylabel=label)
        ax.grid(alpha=0.25)
    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False)
    fig.suptitle(f"{dataset.upper()} — coactivation graph threshold sensitivity", fontsize=15)
    fig.tight_layout(rect=(0, 0.10, 1, 0.95))
    plt.show()

In [ ]:
# save the dataframe to a CSV file
df.to_csv("threshold_analysis_results.csv", index=False)